# Scraping for posts

In [ ]:
import requests
import pandas as pd
import time 
import os
from datetime import datetime

BASE_URL = "https://arctic-shift.photon-reddit.com"
SUBREDDITS = ["BPD","BPDlovedones"]
SEARCH_TERMS = ['"favorite person"','"favourite person"','"my fp"','"fp"']

DATE_START = "2016-01-01"
DATE_END = "2025-12-31"

POST_FIELDS    = "id,subreddit,title,selftext,author,created_utc,score,num_comments"
COMMENT_FIELDS = "id,subreddit,body,author,created_utc,score,link_id"

OUTPUT_DIR = "./data"
SLEEP_BETWEEN_REQUESTS = 1.5

def fetch_page(endpoint: str, params: dict) -> list[dict]:
    url = f"{BASE_URL}{endpoint}"
    try:
        resp = requests.get(url, params=params, timeout=30)
        resp.raise_for_status()
        data = resp.json()
        return data.get("data", [])
    except requests.exceptions.Timeout:
        print(f"  [TIMEOUT] Retrying in 5s...")
        time.sleep(5)
        return []

def collect_all(endpoint: str, base_params: dict, content_type: str) -> list[dict]:
    all_results = []
    params = base_params.copy()
    params["sort"] = "asc"
    params["limit"] = 100
 
    page = 0
    while True:
        page += 1
        print(f"    Fetching page {page} (after={params.get('after', DATE_START)})...")
        results = fetch_page(endpoint, params)
 
        if not results:
            break
 
        all_results.extend(results)
 
        # If fewer than 100 results returned, we've hit the end
        if len(results) < 100:
            break
 
        # Advance cursor to just after the last result's timestamp
        last_ts = results[-1]["created_utc"]
        # Add 1 second to avoid re-fetching the last post
        params["after"] = int(last_ts) + 1
 
        # Safety: stop if we've passed our end date
        if int(last_ts) > int(datetime.strptime(DATE_END, "%Y-%m-%d").timestamp()):
            break
 
        time.sleep(SLEEP_BETWEEN_REQUESTS)
 
    print(f"    → {len(all_results)} {content_type} collected")
    return all_results
 
 
def clean_posts(raw: list[dict], term: str, subreddit: str) -> list[dict]:
    """Normalize raw post dicts into flat records."""
    cleaned = []
    for p in raw:
        ts = p.get("created_utc")
        dt = datetime.utcfromtimestamp(int(ts)) if ts else None
        cleaned.append({
            "id":           p.get("id"),
            "type":         "post",
            "subreddit":    subreddit,
            "search_term":  term,
            "author":       p.get("author"),
            "title":        p.get("title", ""),
            "text":         p.get("selftext", ""),
            "score":        p.get("score"),
            "num_comments": p.get("num_comments"),
            "created_utc":  ts,
            "year":         dt.year if dt else None,
            "month":        dt.month if dt else None,
            "date":         dt.strftime("%Y-%m-%d") if dt else None,
        })
    return cleaned
 
 
def clean_comments(raw: list[dict], term: str, subreddit: str) -> list[dict]:
    """Normalize raw comment dicts into flat records."""
    cleaned = []
    for c in raw:
        ts = c.get("created_utc")
        dt = datetime.utcfromtimestamp(int(ts)) if ts else None
        cleaned.append({
            "id":           c.get("id"),
            "type":         "comment",
            "subreddit":    subreddit,
            "search_term":  term,
            "author":       c.get("author"),
            "title":        "",           # comments have no title
            "text":         c.get("body", ""),
            "score":        c.get("score"),
            "num_comments": None,
            "created_utc":  ts,
            "year":         dt.year if dt else None,
            "month":        dt.month if dt else None,
            "date":         dt.strftime("%Y-%m-%d") if dt else None,
        })
    return cleaned
 
# ── Main ───────────────────────────────────────────────────────────────────────
 
def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)
 
    for subreddit in SUBREDDITS:
        print(f"\n{'='*60}")
        print(f"  Subreddit: r/{subreddit}")
        print(f"{'='*60}")
 
        all_records = []
 
        for term in SEARCH_TERMS:
            print(f"\n  Search term: {term}")
 
            # ── Posts ──────────────────────────────────────────────────────
            print("  [Posts]")
            post_params = {
                "subreddit": subreddit,
                "query":     term,          # searches title + selftext
                "after":     DATE_START,
                "before":    DATE_END,
                "fields":    POST_FIELDS,
            }
            raw_posts = collect_all("/api/posts/search", post_params, "posts")
            all_records.extend(clean_posts(raw_posts, term, subreddit))
 
            time.sleep(SLEEP_BETWEEN_REQUESTS)
 
            # ── Comments ───────────────────────────────────────────────────
            print("  [Comments]")
            comment_params = {
                "subreddit": subreddit,
                "body":      term,
                "after":     DATE_START,
                "before":    DATE_END,
                "fields":    COMMENT_FIELDS,
            }
            raw_comments = collect_all("/api/comments/search", comment_params, "comments")
            all_records.extend(clean_comments(raw_comments, term, subreddit))
 
            time.sleep(SLEEP_BETWEEN_REQUESTS)
 
        # ── Deduplicate ────────────────────────────────────────────────────
        # A post can match multiple search terms; keep one copy per id
        df = pd.DataFrame(all_records)
        before_dedup = len(df)
        df = df.drop_duplicates(subset=["id", "type"])
        after_dedup = len(df)
        print(f"\n  Deduplication: {before_dedup} → {after_dedup} records")
 
        # ── Sort chronologically ───────────────────────────────────────────
        df = df.sort_values("created_utc").reset_index(drop=True)
 
        # ── Save ───────────────────────────────────────────────────────────
        outpath = os.path.join(OUTPUT_DIR, f"{subreddit.lower()}_fp_posts.csv")
        df.to_csv(outpath, index=False, encoding="utf-8")
        print(f"  Saved {len(df)} records → {outpath}")
 
        # Quick summary
        print(f"\n  Year breakdown:")
        if "year" in df.columns and not df["year"].isna().all():
            print(df.groupby("year")["id"].count().to_string())
 
    print(f"\n{'='*60}")
    print("  Collection complete.")
    print(f"  Files saved to: {os.path.abspath(OUTPUT_DIR)}/")
    print(f"{'='*60}\n")
 
if __name__ == "__main__":
    main()